# LogisticRegression

References: [mlu](https://mlu-explain.github.io/logistic-regression/), [google-ml](https://developers.google.com/machine-learning/crash-course/logistic-regression/sigmoid-function), [wiki](https://en.wikipedia.org/wiki/Logistic_regression)

Logistic regression is a supervised machine learning algorithm widely used for binary classification tasks, such as identifying whether an email is spam or not and diagnosing diseases by assessing the presence or absence of specific conditions based on patient test results. This approach utilizes the logistic (or sigmoid) function to transform a linear combination of input features into a probability value ranging between 0 and 1. This probability indicates the likelihood that a given input corresponds to one of two predefined categories. The essential mechanism of logistic regression is grounded in the logistic function's ability to model the probability of binary outcomes accurately. With its distinctive S-shaped curve, the logistic function effectively maps any real-valued number to a value within the 0 to 1 interval. This feature renders it particularly suitable for binary classification tasks, such as sorting emails into "spam" or "not spam". By calculating the probability that the dependent variable will be categorized into a specific group, logistic regression provides a probabilistic framework that supports informed decision-making.


<center><img width="50%" src="https://upload.wikimedia.org/wikipedia/commons/thumb/c/cb/Exam_pass_logistic_curve.svg/3840px-Exam_pass_logistic_curve.svg.png"></center>

<span style="font-weight:bold; background-color: yellow"  > Why it is different from linear regression?</span>

Linear regression outputs unbounded continuous values, like predicting house prices or temperatures. Logistic regression outputs probabilities between 0 and 1, ideal for binary classification (e.g., yes/no, spam/not spam) using a sigmoid function to squash results.

<u>Mathematical Model</u>

Linear regression fits data with a straight line: $Y = a + bX$. Logistic regression applies a logistic (sigmoid) function to the linear equation, modeling log-odds: $P(Y=1) = \frac{1}{1 + e^{-(a + bX)}}$

<span style="font-weight:bold; background-color: yellow"  > what is a sigmoid function ($\sigma$) and why it is  used in Logistic Regression ?</span>

The sigmoid, often called the logistic function, transforms inputs from 
$(-\infty, \infty)$ to $(0,1)$. As $x$ approaches positive infinity, 
$\sigma(x)$ nears $1$; as it approaches negative infinity, it nears $0$, 
with $\sigma(0) = 0.5$.

<center><img width="40%" src="https://upload.wikimedia.org/wikipedia/commons/thumb/8/88/Logistic-curve.svg/3840px-Logistic-curve.svg.png"></center>

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

This smooth curve makes it ideal for modeling probabilities.


where:

- $f(x)$ is the output of the sigmoid function.
- $e$ is Euler's number: a mathematical constant ≈ 2.71828.
- $x$ is the input to the sigmoid function.

Without the sigmoid, linear regression could predict values outside the \([0,1]\) range, which doesn't make sense for classification probabilities (e.g., spam likelihood). The sigmoid "squashes" results into interpretable probabilities while enabling gradient-based optimization via its derivative: $
\sigma'(x) = \sigma(x) \big(1 - \sigma(x)\big)$

### Transforming linear output using the sigmoid function

The following equation represents the linear component of a logistic regression model:

$$z = b + w_1x_1 + w_2x_2 + \ldots + w_Nx_N$$


where:

- z is the output of the linear equation, also called the [log odds](https://en.wikipedia.org/wiki/Logit).

<blockquote>In the equation , z is referred to as the log-odds because if you start with the following sigmoid function (where  is the output of a logistic regression model, representing a probability):

$$ y = \frac{1}{1 + e^{-z}} $$

And then solve for z:

$$ z = \ln\left(\frac{y}{1-y}\right) $$


Then z is defined as the natural logarithm of the ratio of the probabilities of the two possible outcomes: y and 1 – y.</blockquote>
- b is the bias.
- The w values are the model's learned weights.
- The x values are the feature values for a particular example.

To obtain the logistic regression prediction, the z value is then passed to the sigmoid function, yielding a value (a probability) between 0 and 1:

$$ y' = \frac{1}{1 + e^{-z}} $$

where:

- $y'$ is the output of the logistic regression model.
- e is Euler's number: a mathematical constant ≈ 2.71828.
- z is the linear output (as calculated in the preceding equation).

This how linear output is transformed to logistic regression output using these calculations.

<img src="https://developers.google.com/static/machine-learning/crash-course/logistic-regression/images/linear_to_logistic.png">

Logistic regression models are trained using the same process as linear regression models, with two key distinctions:

- Logistic regression models use Log Loss as the loss function instead of squared loss.
- Applying regularization is critical to prevent overfitting.



In [88]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

df = pd.read_csv("../datasets/data.csv")

X = df.drop(columns=['id', 'diagnosis', 'Unnamed: 32']).to_numpy()
y = df['diagnosis']

le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(X_train.shape, y_train.shape)
y_train = y_train.reshape(-1, 1)  # shape (m,1)


(398, 30) (398,)


In [89]:
import numpy as np

class LogisticRegression:
    def __init__(self, learning_rate=0.01, num_iterations=1000):
        self.learning_rate = learning_rate
        self.num_iterations = num_iterations

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X_train, y_train):
        m, n = X_train.shape
        # Initialize weights and bias
        self.W = np.random.randn(n, 1) * 0.01
        self.b = 0

        # Store loss at each iteration for monitoring
        self.loss_history = []

        for i in range(self.num_iterations):
            # Forward pass
            z = X_train @ self.W + self.b
            y_pred = self.sigmoid(z)
            
            # Clip to avoid log(0)
            epsilon = 1e-15
            y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
            
            # Compute loss
            loss = - (1/m) * np.sum(
                y_train * np.log(y_pred) + (1 - y_train) * np.log(1 - y_pred)
            )
            self.loss_history.append(loss)

            # Gradients
            dz = y_pred - y_train  # shape (m,1)
            dW = (1/m) * (X_train.T @ dz)  # shape (n,1)
            db = (1/m) * np.sum(dz)

            # Update weights
            self.W -= self.learning_rate * dW
            self.b -= self.learning_rate * db

            # Optional: print loss every 100 iterations
            if (i+1) % 100 == 0:
                print(f"Iteration {i+1}, Loss: {loss:.4f}")

        return self


model = LogisticRegression()
model.fit(X_train, y_train)




Iteration 100, Loss: 21.6089
Iteration 200, Loss: 8.6624
Iteration 300, Loss: 5.7179
Iteration 400, Loss: 6.2601
Iteration 500, Loss: 5.6502
Iteration 600, Loss: 4.6447
Iteration 700, Loss: 4.1916
Iteration 800, Loss: 3.9902
Iteration 900, Loss: 3.8504
Iteration 1000, Loss: 3.7099


/var/folders/vp/wm4ncjd13qd535sbnxfnl5vh0000gn/T/ipykernel_1439/1129149606.py:9: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))


In [92]:
y_pred = model.sigmoid(X_train @ model.W + model.b) >= 0.5
accuracy = np.mean(y_pred == y_train)
print(f"Training Accuracy: {accuracy*100:.2f}%")

Training Accuracy: 90.45%


/var/folders/vp/wm4ncjd13qd535sbnxfnl5vh0000gn/T/ipykernel_1439/1129149606.py:9: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))


In [93]:
# Ensure y_test is a column vector
y_test = y_test.reshape(-1, 1)  # shape (num_test_samples, 1)

# Compute predicted probabilities
y_prob = model.sigmoid(X_test @ model.W + model.b)  # shape (num_test_samples, 1)

# Convert probabilities to binary class labels (0 or 1)
y_pred = (y_prob >= 0.5).astype(int)

# Calculate accuracy
test_accuracy = np.mean(y_pred == y_test) * 100

print(f"Testing Accuracy: {test_accuracy:.2f}%")

Testing Accuracy: 95.32%


/var/folders/vp/wm4ncjd13qd535sbnxfnl5vh0000gn/T/ipykernel_1439/1129149606.py:9: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-z))
